# Phase 2: GeoJSON streaming export

Adds synthetic lat/lon to the telemetry stream and writes each Spark micro-batch as GeoJSON. Coordinates are simulated, not from GPS.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.spark_setup import configure_spark_environment
from pyspark.sql import SparkSession

java_home = configure_spark_environment(PROJECT_ROOT)
if java_home is None:
    raise RuntimeError(
        "Java not found. Install with: brew install openjdk@17"
    )

spark_builder = (SparkSession.builder
    .appName('Geospatial-Streaming-Export')
    .master('local[*]')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '10'))

if os.name == 'nt':
    spark_builder = spark_builder.config('spark.driver.host', '127.0.0.1')

spark = spark_builder.getOrCreate()
print(f'Project root: {PROJECT_ROOT}')
print(f'JAVA_HOME: {java_home}')

## GeoJSON export

Walk ~1.4 m/s, bike ~4.5 m/s, stand stays put. Direction is based on agent ID.

In [ ]:
from src.geospatial_streaming import run_geospatial_streaming_simulation

output_dir = run_geospatial_streaming_simulation(spark, PROJECT_ROOT)
geojson_files = sorted(output_dir.glob('*.geojson'))
print(f'Created {len(geojson_files)} GeoJSON micro-batch files in {output_dir}')
geojson_files[:3]

In [ ]:
import json

with geojson_files[0].open('r', encoding='utf-8') as file_handle:
    first_batch = json.load(file_handle)

print(first_batch['type'])
print(f"Features in first batch: {len(first_batch['features'])}")
first_batch['features'][0]

## Interpretation and limitation

The generated GeoJSON files demonstrate how a Spark Structured Streaming pipeline can attach a geographical representation to synthetic telemetry. The coordinates are generated from explicit speed and direction assumptions; they are not observed GPS positions and are not snapped to official Vienna transport infrastructure. Consequently, the output is suitable for demonstrating data processing and visualisation, but not for drawing conclusions about actual Vienna mobility or congestion.

In [ ]:
spark.stop()